# 📖 Notebook 2: Rate Limiting & API Key Authentication

Your API is public. Without protection, anyone can flood your services with requests or access data they shouldn't see. This notebook shows how to solve both problems **at the gateway level**.

We'll cover:
- 🚫 **BAD**: No protection at all — services are wide open
- ✅ **BETTER**: Each service implements its own rate limiting with Redis
- 🏆 **BEST**: Centralized rate limiting and API key auth at the gateway

## Learning Objectives

By the end of this notebook, you'll understand:
- Why rate limiting is essential for any public API
- How to implement fixed-window and sliding-window rate limiters using Redis
- Why a fixed window lets a client push **2x** the limit through a window boundary
- Why centralizing rate limiting at the gateway is better than per-service
- How API key authentication works at the gateway level
- How nginx's `limit_req` module works

## 🛠️ Setup

Make sure infrastructure is running:

```bash
cd 05-microservices/api-gateway
docker compose up -d --build
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import requests
import redis
import time
import json
import uuid

GATEWAY = "http://localhost:8080"

def show(response):
    """Print HTTP status and JSON body."""
    print(f"Status: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text[:200])

# Verify services are running
for name, url in [("Gateway", f"{GATEWAY}/health"), ("Redis", None)]:
    if url:
        try:
            r = requests.get(url, timeout=3)
            print(f"✅ {name}: {r.json()['status']}")
        except Exception as e:
            print(f"❌ {name}: {e}")
    else:
        try:
            rc = redis.Redis(host="localhost", port=6380, decode_responses=True)
            rc.ping()
            print(f"✅ Redis: connected (port 6380)")
        except Exception as e:
            print(f"❌ Redis: {e}")

---

## 🚫 BAD: No Rate Limiting

Without rate limiting, any client can send as many requests as they want. This leads to:
- **DDoS vulnerability** — a single bad actor can take down your service
- **Resource exhaustion** — one heavy user consumes all capacity
- **No fairness** — well-behaved clients get worse performance

Let's see what happens when we flood a service directly:

In [ ]:
# BAD: Flooding a service directly — no protection at all

from concurrent.futures import ThreadPoolExecutor

def flood_service(url, num_requests=50, workers=10):
    """Send many requests as fast as possible."""

    def make_request(_):
        # Return the outcome instead of mutating a shared dict: `d[k] += 1`
        # is load-add-store, so N threads doing it can silently lose counts.
        try:
            return requests.get(url, timeout=5).status_code == 200
        except Exception:
            return False

    start = time.time()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        outcomes = list(pool.map(make_request, range(num_requests)))
    elapsed = time.time() - start

    ok = sum(outcomes)
    return {"success": ok, "failed": len(outcomes) - ok}, elapsed

print("🚫 BAD: Flooding the User Service directly (no rate limiting)")
print("=" * 60)
print()

results, elapsed = flood_service("http://localhost:5001/users", num_requests=50)

print(f"Sent 50 requests in {elapsed:.2f} seconds")
print(f"  ✅ Successful: {results['success']}")
print(f"  ❌ Failed:     {results['failed']}")
print(f"  Rate:          {50/elapsed:.0f} requests/second")
print()
print("⚠️  ALL requests succeeded! The service has no protection.")
print("   A malicious actor could easily overwhelm it.")

# This is the whole point of the "BAD" section: an unprotected service
# accepts everything you throw at it. If it ever starts rejecting requests,
# something in front of it is doing rate limiting and this demo is a lie.
assert results["success"] == 50, (
    f"the unprotected service should accept all 50 requests, got "
    f"{results['success']} ok / {results['failed']} failed")

---

## ✅ BETTER: Per-Service Rate Limiting with Redis

One approach is to add rate limiting **inside each service**. We'll use Redis to track request counts because:

- Redis is **fast** (in-memory) — it adds almost no latency to the request path
- Redis is **shared** — the counter works across every instance of the service
- Redis has **built-in TTL** — counters expire on their own, so nothing to clean up

### How a Fixed-Window Rate Limiter Works

Time is chopped into equal, non-overlapping buckets (e.g. 10-second windows). Every request increments the counter for the current bucket; when the bucket rolls over, the counter resets to zero.

```
                |------- window N -------||------ window N+1 ------|
limit: 5/window
requests:        R  R  R  R  R   R   R      R  R  R  R  R    R
counter:         1  2  3  4  5   6   7      1  2  3  4  5    6
verdict:         ok ok ok ok ok  X   X      ok ok ok ok ok   X
```

For each request:

1. Compute the current bucket: `bucket = floor(now / window_seconds)`
2. Build a Redis key like `rate_limit:<client_id>:<bucket>`
3. Increment the counter with `INCR` (atomic — no read-modify-write race)
4. On the first request in the window, set a TTL so the key expires automatically
5. If the counter is now greater than the limit, reject the request (HTTP 429)

> **Fixed vs. sliding window.** Fixed-window is the simplest algorithm and it's what we implement below. Its weakness is the *burst-at-the-boundary* problem: a client can send `limit` requests at the very end of one window and `limit` more at the start of the next, pushing **2x the limit** through in a span shorter than a single window. We're not going to just assert that — the next two cells **reproduce it**, then fix it with a true sliding window.

In [ ]:
# BETTER: Implement a rate limiter using Redis
# This is what each service would have to implement individually

r = redis.Redis(host="localhost", port=6380, decode_responses=True)

def check_rate_limit(client_id: str, max_requests: int = 5, window_seconds: int = 10) -> dict:
    """
    Fixed-window rate limiter using Redis.
    
    Args:
        client_id: Identifies the client (usually IP address or API key)
        max_requests: Maximum requests allowed per window
        window_seconds: Length of the time window in seconds
    
    Returns:
        dict with 'allowed' (bool) and metadata
    """
    # Current time window (e.g., requests in the same 10-second block)
    current_window = int(time.time()) // window_seconds
    key = f"rate_limit:{client_id}:{current_window}"
    
    # Increment the counter for this window
    current_count = r.incr(key)
    
    # Set expiry on first request in this window
    if current_count == 1:
        r.expire(key, window_seconds)
    
    allowed = current_count <= max_requests
    
    return {
        "allowed": allowed,
        "window": current_window,   # which bucket this request landed in
        "current_count": current_count,
        "max_requests": max_requests,
        "remaining": max(0, max_requests - current_count),
        "window_seconds": window_seconds
    }

print("✅ BETTER: Redis-based rate limiter")
print("=" * 55)
print("Limit: 5 requests per 10-second window")
print()

# Simulate 8 requests from the same client.
# We retry if the batch happens to straddle a window rollover -- the whole
# point of THIS cell is one clean window; the boundary case gets its own
# section below. Old keys are dropped first so re-running the cell is
# reproducible instead of continuing a half-full counter.
for _ in range(3):
    for stale in r.scan_iter("rate_limit:test-client:*"):
        r.delete(stale)
    verdicts = [check_rate_limit("test-client", max_requests=5, window_seconds=10)
                for _ in range(8)]
    if len({v["window"] for v in verdicts}) == 1:
        break

for i, result in enumerate(verdicts):
    status = "✅ Allowed" if result["allowed"] else "❌ BLOCKED"
    print(f"  Request {i+1}: {status}  (count: {result['current_count']}/{result['max_requests']}, remaining: {result['remaining']})")

# Inside a single window the split must be exactly 5 allowed / 3 blocked.
assert len({v["window"] for v in verdicts}) == 1, "all 8 requests should share one bucket"
assert [v["allowed"] for v in verdicts] == [True] * 5 + [False] * 3, (
    f"expected 5 allowed then 3 blocked, got {[v['allowed'] for v in verdicts]}")

print()
print("💡 After 5 requests, additional ones are blocked until the window resets.")
print()
print("⚠️  Problem: Every service needs this same code!")
print("   User Service has it, Order Service has it, Payment Service has it...")
print("   That's duplicated logic and inconsistent limits.")

### 🔥 Reproducing the burst-at-the-boundary bug

Prose about a failure mode is cheap. Let's actually make a fixed-window limiter
let **twice** its configured limit through.

The attack is trivial: line up on a window boundary, fire `limit` requests at the
tail of window *N*, wait for the clock to tick over, fire `limit` more at the head
of window *N+1*. Both batches are individually legal. Together they are `2 x limit`
requests inside a span far shorter than one window — which is exactly the thing the
limiter was supposed to prevent.

```
              window N                 window N+1
   ......................|.........................
                    RRRRR|RRRRR
                     ^^^^^ ^^^^^
                 5 allowed  5 allowed   -> 10 requests in ~0.3s
                            with a limit of 5 per 4s
```

In [ ]:
# Reproduce the bug: 2x the limit through a fixed window boundary.

BURST_LIMIT = 5    # requests allowed per window
BURST_WINDOW = 4   # seconds per window


def clear_keys(pattern):
    for key in r.scan_iter(pattern):
        r.delete(key)


def straddle_the_boundary(limiter, client_id, attempts=6):
    """Fire BURST_LIMIT requests just before a window rollover and BURST_LIMIT just after.

    Returns (verdicts, elapsed_seconds). We retry a few times because we are
    racing a wall clock: if Redis hiccups, a batch can drift into the wrong
    bucket, and then we would be measuring nothing.
    """
    for _ in range(attempts):
        clear_keys(f"rate_limit:{client_id}:*")
        clear_keys(f"sliding:{client_id}")

        now = time.time()
        boundary = (int(now) // BURST_WINDOW + 1) * BURST_WINDOW  # next rollover
        time.sleep(max(0.0, boundary - 0.30 - now))               # wake 300ms early

        t0 = time.time()
        before = [limiter(client_id, BURST_LIMIT, BURST_WINDOW) for _ in range(BURST_LIMIT)]

        time.sleep(max(0.0, (boundary + 0.05) - time.time()))     # step just past it
        after = [limiter(client_id, BURST_LIMIT, BURST_WINDOW) for _ in range(BURST_LIMIT)]
        elapsed = time.time() - t0

        # Did we actually land the two batches on opposite sides of a rollover?
        if int(t0) // BURST_WINDOW != int(time.time()) // BURST_WINDOW:
            return before + after, elapsed
    raise RuntimeError("could not line up on a window boundary -- is Redis slow?")


print("🔥 Fixed window: burst at the boundary")
print("=" * 55)
print(f"Limit: {BURST_LIMIT} requests per {BURST_WINDOW}s window")
print()

verdicts, elapsed = straddle_the_boundary(check_rate_limit, "boundary-attacker")
allowed = sum(v["allowed"] for v in verdicts)

for i, v in enumerate(verdicts):
    side = "before rollover" if i < BURST_LIMIT else "after rollover "
    mark = "✅ Allowed" if v["allowed"] else "❌ BLOCKED"
    print(f"  req {i+1:>2} ({side}): {mark}  bucket={v['window']} count={v['current_count']}")

print()
print(f"  Requests allowed: {allowed}  in {elapsed:.2f}s")
print(f"  Configured limit: {BURST_LIMIT} per {BURST_WINDOW}s")
print(f"  Effective rate:   {allowed / elapsed:.0f} req/s "
      f"(vs {BURST_LIMIT / BURST_WINDOW:.2f} req/s intended)")
print()
print("💥 Every single request was 'legal' by the limiter's own rules, and the")
print("   client still got 2x the quota through in a fraction of one window.")

# This cell only teaches something if the bug actually reproduces.
assert allowed == 2 * BURST_LIMIT, (
    f"the fixed window should have let 2x the limit through at the boundary, "
    f"but only {allowed}/{2 * BURST_LIMIT} requests were allowed")
assert elapsed < BURST_WINDOW, (
    f"the burst must fit inside ONE window to be a violation, took {elapsed:.2f}s")

### ✅ The fix: a true sliding window

The bug is that a fixed window has **amnesia**: the moment the clock ticks over,
the previous 5 requests stop existing. A sliding window instead asks *"how many
requests happened in the last N seconds, counted from right now?"* — so there is
no boundary to hide behind.

The classic implementation is a **sliding window log**: keep one Redis sorted-set
entry per admitted request, scored by timestamp.

1. `ZREMRANGEBYSCORE key 0 (now - window)` — evict everything that aged out
2. `ZCARD key` — how many requests are still inside the window
3. If that count is below the limit, `ZADD` this request and allow it; otherwise reject
4. `EXPIRE` the key so an idle client's data disappears

Steps 1–3 must be **one atomic operation**, otherwise two concurrent requests both
read "4 requests so far" and both admit themselves. Doing it in a Python
`if` around three separate Redis calls is a race, not a rate limiter — so we ship
it as a Lua script, which Redis runs to completion without interleaving. This is
exactly how production limiters (and nginx's own `limit_req`) get correctness.

**Cost:** one sorted-set entry per admitted request per client, instead of one
integer per window. That's the trade — precision costs memory. A *sliding window
counter* (weighted blend of the previous and current fixed windows) is the usual
middle ground: near-sliding accuracy at fixed-window memory cost.

In [ ]:
# Sliding-window-log limiter. Same interface as check_rate_limit, no boundary to exploit.

_SLIDING_LUA = """
local key    = KEYS[1]
local now    = tonumber(ARGV[1])
local window = tonumber(ARGV[2])
local limit  = tonumber(ARGV[3])
local member = ARGV[4]

-- 1. evict everything that has aged out of the window
redis.call('ZREMRANGEBYSCORE', key, 0, now - window)
-- 2. how many requests are still inside it?
local count = redis.call('ZCARD', key)
-- 3. admit only if there is room (and only then does it occupy a slot)
local allowed = 0
if count < limit then
    redis.call('ZADD', key, now, member)
    count = count + 1
    allowed = 1
end
-- 4. let an idle client's key disappear on its own
redis.call('EXPIRE', key, window + 1)
return {allowed, count}
"""

_sliding_script = r.register_script(_SLIDING_LUA)


def check_sliding_window(client_id: str, max_requests: int = 5, window_seconds: int = 10) -> dict:
    """Sliding-window-log rate limiter. Counts requests in the last `window_seconds`."""
    now = time.time()
    # The member must be unique per request -- two requests in the same
    # microsecond would otherwise collapse into one sorted-set entry.
    member = f"{now:.6f}:{uuid.uuid4().hex[:8]}"
    allowed, count = _sliding_script(
        keys=[f"sliding:{client_id}"],
        args=[now, window_seconds, max_requests, member],
    )
    return {
        "allowed": bool(allowed),
        "window": "sliding",
        "current_count": count,
        "max_requests": max_requests,
        "remaining": max(0, max_requests - count),
        "window_seconds": window_seconds,
    }


print("✅ Sliding window: same attack, same boundary")
print("=" * 55)
print(f"Limit: {BURST_LIMIT} requests per {BURST_WINDOW}s (rolling)")
print()

verdicts_sw, elapsed_sw = straddle_the_boundary(check_sliding_window, "boundary-attacker")
allowed_sw = sum(v["allowed"] for v in verdicts_sw)

for i, v in enumerate(verdicts_sw):
    side = "before rollover" if i < BURST_LIMIT else "after rollover "
    mark = "✅ Allowed" if v["allowed"] else "❌ BLOCKED"
    print(f"  req {i+1:>2} ({side}): {mark}  in-window count={v['current_count']}")

print()
print(f"  Fixed window   → {2 * BURST_LIMIT} allowed in {elapsed:.2f}s   (limit was {BURST_LIMIT})")
print(f"  Sliding window → {allowed_sw} allowed in {elapsed_sw:.2f}s   (limit was {BURST_LIMIT})")
print()
print("💡 The rollover means nothing to a sliding window -- the first 5 requests")
print("   are still inside the last 4 seconds, so the next 5 are refused.")

assert allowed_sw == BURST_LIMIT, (
    f"the sliding window must cap the same burst at {BURST_LIMIT}, got {allowed_sw}")
assert allowed_sw < sum(v["allowed"] for v in verdicts), (
    "the sliding window should admit strictly fewer requests than the fixed window here")

In [ ]:
# Clean up the rate limit keys from our demos (both algorithms)
removed = 0
for pattern in ("rate_limit:*", "sliding:*"):
    for key in r.scan_iter(pattern):
        r.delete(key)
        removed += 1
print(f"🧹 Cleaned up {removed} Redis rate limit keys")

---

## 🏆 BEST: Gateway-Level Rate Limiting

Instead of each service implementing rate limiting, the **API gateway** does it for everyone:

```
┌──────────┐     ┌──────────────────────────┐     ┌─────────────┐
│  Client   │────▶│  API Gateway              │────▶│  Backend    │
│           │     │  ┌──────────────────────┐ │     │  Service    │
│           │     │  │ Rate Limiter          │ │     │             │
│           │     │  │ Too many? → 429 ❌    │ │     │             │
│           │     │  │ OK?       → forward ✅│ │     │             │
│           │     │  └──────────────────────┘ │     │             │
└──────────┘     └──────────────────────────┘     └─────────────┘
```

Benefits:
- **One place** to configure limits for all services
- **Consistent** behavior across all endpoints
- **Blocked requests never reach backends** — saving resources

Our nginx config uses the `limit_req` module:

```nginx
# Define the rate limit zone (in the http block)
limit_req_zone $binary_remote_addr zone=api_limit:10m rate=5r/s;

# Apply it to a location
location /api/users {
    limit_req zone=api_limit burst=10 nodelay;
    limit_req_status 429;
    proxy_pass http://user_backend/users;
}
```

- `rate=5r/s` — allow 5 requests per second per IP
- `burst=10` — allow short bursts up to 10 extra requests
- `nodelay` — don't queue burst requests, serve them immediately
- `limit_req_status 429` — return HTTP 429 (Too Many Requests) when blocked

### Do the arithmetic before you run the cell

`limit_req` is a **leaky bucket**, not a fixed window. nginx tracks an `excess`
counter per IP that drains at exactly `rate` and is topped up by each request.
A request is served while `excess <= burst`. So from an idle start:

```
instantaneous capacity = 1 (the request that finds excess = 0) + burst
                       = 1 + 10
                       = 11 requests
```

and after that, capacity refills at `rate`, i.e. one more slot every 200ms.
If the whole loop takes `T` seconds, expect roughly `11 + 5*T` successes and the
rest 429. **Predict the number first, then check it against the output** — the
cell asserts this, so if nginx ever behaves differently you'll find out loudly.

Note the difference from the Redis limiter above: a leaky bucket has no window
boundary to straddle, which is why it doesn't suffer the 2x burst bug.

In [ ]:
# BEST: Let's trigger nginx's rate limiter by sending rapid requests

RATE, BURST, N = 5, 10, 30

print("🏆 BEST: Gateway-Level Rate Limiting (nginx limit_req)")
print("=" * 60)
print(f"Config: rate={RATE}r/s, burst={BURST}")
print()

# Start from a known state: sleep long enough for the bucket to drain fully,
# otherwise leftover excess from an earlier cell makes the numbers unreadable.
time.sleep(BURST / RATE + 1)

# Send requests as fast as possible through the gateway
results = {"200": 0, "429": 0, "other": 0}
statuses = []

t0 = time.time()
for i in range(N):
    r = requests.get(f"{GATEWAY}/api/users")
    statuses.append(r.status_code)
    if r.status_code == 200:
        results["200"] += 1
    elif r.status_code == 429:
        results["429"] += 1
    else:
        results["other"] += 1
loop_seconds = time.time() - t0

print(f"Results from {N} rapid requests (loop took {loop_seconds:.2f}s):")
print(f"  ✅ 200 (OK):              {results['200']}")
print(f"  ❌ 429 (Too Many):        {results['429']}")
if results["other"]:
    print(f"  ⚠️  Other:                {results['other']}")
print()

# Show the pattern
print("Request-by-request status codes:")
line = ""
for i, s in enumerate(statuses):
    line += "✅" if s == 200 else "❌"
    if (i + 1) % 15 == 0:
        print(f"  {line}")
        line = ""
if line:
    print(f"  {line}")

# 1 + burst pass instantly, then the bucket refills at `rate` while we loop.
lower = 1 + BURST
upper = 1 + BURST + int(RATE * loop_seconds) + 1
print()
print(f"Predicted successes: {lower} (1 + burst) .. {upper} (+ refill during {loop_seconds:.2f}s)")
print(f"Actual successes:    {results['200']}")

assert results["other"] == 0, f"unexpected status codes: {set(statuses)}"
assert lower <= results["200"] <= upper, (
    f"limit_req should admit 1+burst={lower} plus refill (<= {upper}), "
    f"got {results['200']} in {loop_seconds:.2f}s")
assert results["429"] > 0, "the rate limiter never fired -- this demo proves nothing"

print()
print("💡 The gateway blocks excess requests BEFORE they reach the backend.")
print("   Your services are protected without writing a single line of rate limiting code!")

In [ ]:
# Wait for the rate limit window to reset, then show normal behavior

print("⏳ Waiting 3 seconds for the rate limit window to reset...")
time.sleep(3)

print()
print("Now sending 5 requests at a normal pace (1 per second):")
for i in range(5):
    r = requests.get(f"{GATEWAY}/api/users")
    print(f"  Request {i+1}: Status {r.status_code} {'✅' if r.status_code == 200 else '❌'}")
    time.sleep(0.3)

print()
print("💡 At a normal rate, all requests succeed. Rate limiting only kicks in")
print("   when a client sends too many requests too fast.")

---

## 🔐 API Key Authentication at the Gateway

Rate limiting controls **how much** clients can use the API.  
Authentication controls **who** can use the API.

Our gateway checks for a valid `X-API-Key` header:

```nginx
# Define valid API keys
map $http_x_api_key $api_key_valid {
    default           0;     # Unknown keys → invalid
    "demo-key-123"    1;     # Known keys → valid
    "premium-key-456" 1;
    "admin-key-789"   1;
}

# Protected endpoint — check API key before forwarding
location /api/auth/users {
    if ($api_key_valid = 0) {
        return 401 '{"error": "Unauthorized"}';
    }
    proxy_pass http://user_backend/users;
}
```

Invalid requests are rejected **at the gateway** — they never reach the backend.

In [ ]:
# API Key Authentication demo

time.sleep(1)  # Brief pause to avoid rate limiting from previous demo

print("🔐 API Key Authentication at the Gateway")
print("=" * 55)
print()

# 1. No API key → 401 Unauthorized
print("1️⃣  Request WITHOUT API key:")
r = requests.get(f"{GATEWAY}/api/auth/users")
print(f"   Status: {r.status_code}")
print(f"   Body:   {r.text[:100]}")
print()

# 2. Invalid API key → 401 Unauthorized
print("2️⃣  Request with INVALID API key:")
r = requests.get(f"{GATEWAY}/api/auth/users", headers={"X-API-Key": "fake-key-000"})
print(f"   Status: {r.status_code}")
print(f"   Body:   {r.text[:100]}")
print()

# 3. Valid API key → 200 OK
print("3️⃣  Request with VALID API key:")
r = requests.get(f"{GATEWAY}/api/auth/users", headers={"X-API-Key": "demo-key-123"})
print(f"   Status: {r.status_code}")
data = r.json()
print(f"   Users:  {data['count']} users returned")
print(f"   Served: {data['served_by']}")
print()

print("💡 The backend service never saw the rejected requests!")
print("   Auth is handled entirely at the gateway level.")

In [ ]:
# Show the difference: open endpoint vs authenticated endpoint

time.sleep(1)

print("📊 Comparing Open vs Authenticated Endpoints")
print("=" * 55)
print()

# Open endpoint — no auth needed
print("Open endpoint (/api/users):")
r = requests.get(f"{GATEWAY}/api/users")
print(f"  No key needed → Status: {r.status_code} ✅")
print()

# Authenticated endpoint — same data, but requires a key
print("Authenticated endpoint (/api/auth/users):")
r = requests.get(f"{GATEWAY}/api/auth/users")
print(f"  No key        → Status: {r.status_code} ❌")
r = requests.get(f"{GATEWAY}/api/auth/users", headers={"X-API-Key": "demo-key-123"})
print(f"  With key      → Status: {r.status_code} ✅")
print()

print("💡 In practice, you'd use different paths or the same path with")
print("   auth always required. We use separate paths here for clarity.")
print()
print("🏢 Real-world API key management:")
print("   - Keys stored in a database, not nginx config")
print("   - Different keys get different rate limits (tiers)")
print("   - Keys can be revoked without restarting the gateway")
print("   - JWT tokens are common for user-level auth")

## 📚 Summary

### What We Learned

| Approach | Rate Limiting | Auth | Where |
|----------|:------------:|:----:|-------|
| 🚫 BAD | ❌ None | ❌ None | — |
| ✅ BETTER | ✅ Per-service (Redis) | ✅ Per-service | Each backend service |
| 🏆 BEST | ✅ Centralized (nginx) | ✅ Centralized (nginx) | API Gateway |

### Key Takeaways

1. **Rate limiting is essential** — without it, one client can take down your service
2. **Redis is great for rate limiting** — fast, shared, with built-in expiry
3. **Fixed windows leak 2x at the boundary** — we reproduced it, then fixed it with an atomic sliding-window log in Lua. Match the algorithm to what the limit protects
4. **Do the counter update atomically** — `INCR`, a Lua script, or nginx's own shared-memory zone. A check-then-write in application code is a race, not a limit
5. **Gateway-level is best** — one config protects all services consistently
6. **Blocked requests never reach backends** — saving compute resources
7. **API keys at the gateway** — authenticate before routing, not after

### Rate Limiting Algorithms

| Algorithm | How It Works | Cost | Boundary burst? |
|-----------|-------------|------|-----------------|
| **Fixed Window** | Count requests in fixed time blocks | 1 integer per client per window | ❌ **Yes — we reproduced 2x the limit above** |
| **Sliding Window Log** | Keep a timestamp per admitted request, count the last N seconds | 1 entry per admitted request | ✅ No |
| **Sliding Window Counter** | Weighted blend of previous + current fixed window | 2 integers per client | ✅ Nearly — small approximation error |
| **Token Bucket** | Tokens replenish at a fixed rate, a request spends one | 2 numbers per client | ✅ No (bursts are bounded by bucket size) |
| **Leaky Bucket** | Requests drain at a constant rate; `nginx limit_req` is this one | 2 numbers per client | ✅ No |

Rule of thumb: fixed window is fine when the limit is a rough courtesy (e.g. 1000/hour)
and dangerous when it protects something that actually falls over at 2x load.

### Interview Tip

> When asked about rate limiting, mention: *"I'd implement rate limiting at the API gateway using a sliding-window counter in Redis, evaluated atomically in a Lua script. Fixed windows are simpler but let a client push 2x the limit through a window boundary, which matters when the limit is protecting a capacity ceiling rather than being a courtesy. Doing it at the gateway gives centralized control and per-client tracking with no backend code changes."*

### Next Up

In **Notebook 3**, we'll explore **request transformation** — how the gateway can modify requests and responses, inject headers, and handle API versioning.